# R2-Q2 reference masks — mentor build notebook

This notebook produces the five hand-segmented leaf masks that ship
with `irilab2026` for R2-Q2 Notebook 04's segmentation validation.
It's a one-time mentor task — students don't run it.

Workflow:

1. **Run the proposal cell** to get 5 candidate filenames spanning
   the working-set hosts. Try different `SEED` values until you have
   a set you like.
2. **Run the preview cell** to eyeball the 5 source images and
   confirm they're good hand-segmentation targets.
3. **For each filename, run its polygon-picker cell**, draw the leaf
   boundary, and confirm the captured polygon.
4. **Run the save cell** to write the PNGs and `manifest.csv` to
   the package's data directory.
5. **Commit and push** the new files to the `irilab2026` repo.

Colab caveat: matplotlib's interactive widgets need `ipympl` and
the `%matplotlib widget` backend. If the polygon picker doesn't
respond to clicks, restart the runtime, re-run the setup cell, and
try again. If it still doesn't work, fall back to GIMP or a
polygon-drawing web tool — the save cell is independent of how the
polygons were drawn.

In [ ]:
# Setup: import the same shared machinery N04 uses, plus the
# interactive matplotlib backend for the polygon picker.
!pip install -q ipympl
%matplotlib widget

In [ ]:
import irilab2026 as iri
iri.setup(gpu_required=False)

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from PIL import Image, ImageDraw

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

# Load the working set and (post-N04-Section-3) leaf masks metadata.
working_set = pd.read_parquet(OUTPUT_DIR / "working_set.parquet")
leaf_masks_metadata = pd.read_parquet(OUTPUT_DIR / "leaf_masks_metadata.parquet")

# PD image loader (same shape as N04).
pd_metadata, pd_hf_dataset = iri.load_plantdoc()
filename_to_pd_index = dict(zip(pd_metadata["filename"], pd_metadata.index))


def load_pd_image(filename):
    pd_idx = filename_to_pd_index[filename]
    return pd_hf_dataset[int(pd_idx)]["image"].convert("RGB")


print(f"Working set: {len(working_set)} rows")
print(f"Leaf masks metadata: {len(leaf_masks_metadata)} rows")

In [ ]:
### Propose 5 reference-mask candidates spanning the working-set hosts
### (seeded variant — change SEED to see different picks)

# Reproducibility seed. Same SEED + same working set + same leaf_masks_metadata
# = same five picks. Write this seed down alongside the picks you finally
# accept, so a future iteration can recover them.
SEED = 12

# Join per-image metadata with host and category for stratification.
candidates = leaf_masks_metadata.merge(
    working_set[["filename", "host", "true_category"]],
    on="filename",
    how="left",
)

# Filter to the mid-band of fg_fractions. The hand-segmentation task is
# most meaningful on images where the leaf is a clearly visible subject —
# not a tiny dot or a near-full-frame blob.
MID_LOW, MID_HIGH = 0.10, 0.85
mid_band = candidates[
    (candidates["fg_fraction"] >= MID_LOW)
    & (candidates["fg_fraction"] <= MID_HIGH)
]

# Top 5 hosts by working-set count. This part is still deterministic —
# the same five hosts come back every run for a given working set.
host_counts = mid_band["host"].value_counts()
print(f"Candidate distribution across hosts (mid-band only):")
print(host_counts.to_string())
print()

top_5_hosts = host_counts.head(5).index.tolist()

# For each host, take one random sample from that host's mid-band pool,
# keyed by SEED. Change SEED to see a different set of 5 picks.
proposals = []
for host in top_5_hosts:
    host_pool = mid_band[mid_band["host"] == host]
    pick = host_pool.sample(1, random_state=SEED).iloc[0]
    proposals.append({
        "filename": pick["filename"],
        "host": pick["host"],
        "true_category": pick["true_category"],
        "fg_fraction": pick["fg_fraction"],
    })

proposals_df = pd.DataFrame(proposals)
print(f"Proposed 5 reference filenames (SEED={SEED}):")
print(proposals_df.to_string(index=False))
print()
print(f"To see a different set of picks, change SEED at the top of this cell")
print(f"and re-run. To commit to these picks, copy the list below into")
print(f"REFERENCE_FILENAMES in the next cell:")
print()
print(proposals_df["filename"].tolist())

In [ ]:
### Propose 5 reference-mask candidates spanning the working-set hosts
### (seeded variant — change SEED to see different picks)

# Reproducibility seed. Same SEED + same working set + same leaf_masks_metadata
# = same five picks. Write this seed down alongside the picks you finally
# accept, so a future iteration can recover them.
SEED = 12

# Join per-image metadata with host and category for stratification.
candidates = leaf_masks_metadata.merge(
    working_set[["filename", "host", "true_category"]],
    on="filename",
    how="left",
)

# Filter to the mid-band of fg_fractions. The hand-segmentation task is
# most meaningful on images where the leaf is a clearly visible subject —
# not a tiny dot or a near-full-frame blob.
MID_LOW, MID_HIGH = 0.10, 0.85
mid_band = candidates[
    (candidates["fg_fraction"] >= MID_LOW)
    & (candidates["fg_fraction"] <= MID_HIGH)
]

# Top 5 hosts by working-set count. This part is still deterministic —
# the same five hosts come back every run for a given working set.
host_counts = mid_band["host"].value_counts()
print(f"Candidate distribution across hosts (mid-band only):")
print(host_counts.to_string())
print()

top_5_hosts = host_counts.head(5).index.tolist()

# For each host, take one random sample from that host's mid-band pool,
# keyed by SEED. Change SEED to see a different set of 5 picks.
proposals = []
for host in top_5_hosts:
    host_pool = mid_band[mid_band["host"] == host]
    pick = host_pool.sample(1, random_state=SEED).iloc[0]
    proposals.append({
        "filename": pick["filename"],
        "host": pick["host"],
        "true_category": pick["true_category"],
        "fg_fraction": pick["fg_fraction"],
    })

proposals_df = pd.DataFrame(proposals)
print(f"Proposed 5 reference filenames (SEED={SEED}):")
print(proposals_df.to_string(index=False))
print()
print(f"To see a different set of picks, change SEED at the top of this cell")
print(f"and re-run. To commit to these picks, copy the list below into")
print(f"REFERENCE_FILENAMES in the next cell:")
print()
print(proposals_df["filename"].tolist())

In [ ]:
# Committed picks from SEED=12. To re-derive: run Cell 3 with SEED=12.
REFERENCE_FILENAMES = [
    '07.17.18-Common_Tomato_Diseases_Canker-258x300.jpg',
    'pepper-spot.jpg',
    '730-grape-leaf-2560x1600-nature-wallpaper.jpg',
    'leaf-blueberry-15281271.jpg',
    '20180511_090912-14gtw8a-e1526047952754.jpg',
]

# Per-pick notes for the manifest. Include the provenance (SEED=12)
# so a future iteration can recover the picks deterministically.
REFERENCE_NOTES = [
    "Tomato host; bacterial; fg=44%; SEED=12",
    "Bell pepper host; bacterial; fg=70%; SEED=12",
    "Grape host; healthy; fg=30%; SEED=12",
    "Blueberry host; healthy; fg=67%; SEED=12",
    "Apple host; healthy; fg=18%; SEED=12",
]

assert len(REFERENCE_FILENAMES) == 5, "Need exactly 5 filenames"
assert len(REFERENCE_NOTES) == 5, "Need exactly 5 notes"
assert all(
    fn in working_set["filename"].values for fn in REFERENCE_FILENAMES
), "Every reference filename must be in the working set"

# Storage for the polygons the mentor draws.
collected_polygons = {}

In [ ]:
### Preview the 5 candidate images before drawing polygons

# Pull host, true_category, and SAM's fg_fraction for each pick — the
# same fields that drove the proposal — so you can sanity-check that
# the picks span the variation you wanted them to span.
preview_meta = (
    working_set[["filename", "host", "true_category"]]
    .merge(
        leaf_masks_metadata[["filename", "fg_fraction"]],
        on="filename",
    )
    .set_index("filename")
)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, filename in enumerate(REFERENCE_FILENAMES):
    image_np = np.array(load_pd_image(filename))
    row = preview_meta.loc[filename]

    axes[i].imshow(image_np)
    axes[i].axis("off")
    axes[i].set_title(
        f"Pick {i+1}: {row['host']}\n"
        f"{row['true_category']} · "
        f"fg={row['fg_fraction']:.0%}\n"
        f"{filename[:30]}...",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

print("If any pick looks like a bad hand-segmentation target —")
print("multiple leaves at similar depth, very blurry, occluded by hands,")
print("or just visually ambiguous — go back to Cell 4 and substitute it")
print("before drawing polygons.")

In [ ]:
def pick_polygon(filename):
    """Display the image and attach a polygon picker.

    The mentor clicks around the leaf boundary; double-click closes
    the polygon. The captured vertices are stored in
    collected_polygons[filename]. The return value is the selector
    object — Python needs a reference to it, otherwise the widget is
    garbage-collected and stops responding.
    """
    image_pil = load_pd_image(filename)
    image_np = np.array(image_pil)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image_np)
    ax.set_title(
        f"{filename}\n"
        "Click around the leaf boundary. "
        "Double-click to close the polygon."
    )
    ax.axis("off")

    def onselect(verts):
        collected_polygons[filename] = verts
        print(
            f"  Captured {len(verts)} vertices for {filename}. "
            f"Move to the next image."
        )

    selector = PolygonSelector(ax, onselect)
    plt.show()
    return selector